# Data Cleaning & Preparation

**Goal:** Transform the raw dataset into a clean, structured, analysis-ready format.

**Input:** Raw dataset via `datasets.load_dataset('lukebarousse/data_jobs')`

**Output:** `data_jobs_clean.parquet`, `data_jobs_salary.parquet` (saved to `data/clean/`)

**What this notebook covers:**
- Parsing nested skill columns
- Fixing data types
- Handling missing values with bias checks
- Isolating a dedicated salary dataset
- Validating the remote work flag

---

## Importing Libraries and Loading Data

In [1]:
# Importing Libraries
import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

# Loading Data
dataset = load_dataset('lukebarousse/data_jobs')
df = dataset['train'].to_pandas()

# First look at the data
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 785741 entries, 0 to 785740
Data columns (total 17 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   job_title_short        785741 non-null  object 
 1   job_title              785740 non-null  object 
 2   job_location           784696 non-null  object 
 3   job_via                785733 non-null  object 
 4   job_schedule_type      773074 non-null  object 
 5   job_work_from_home     785741 non-null  bool   
 6   search_location        785741 non-null  object 
 7   job_posted_date        785741 non-null  object 
 8   job_no_degree_mention  785741 non-null  bool   
 9   job_health_insurance   785741 non-null  bool   
 10  job_country            785692 non-null  object 
 11  salary_rate            33067 non-null   object 
 12  salary_year_avg        22003 non-null   float64
 13  salary_hour_avg        10662 non-null   float64
 14  company_name           785723 non-nu

## Data Cleaning 

### Parsing 'job_skills' and 'job_type_skills' columns 
These columns are stored as string representations of lists and I am converting them to actual Python lists for analysis.

In [2]:
df['job_skills'] = df['job_skills'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else x)
df['job_type_skills'] = df['job_type_skills'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else x)

In [3]:
#Checking if it's all done 
print(type(df['job_skills'].dropna().iloc[0]))
print(type(df['job_type_skills'].dropna().iloc[0]))

<class 'list'>
<class 'dict'>


### Converting 'job_posted_date' to datetime

In [4]:
df['job_posted_date'] = pd.to_datetime(df['job_posted_date'])

In [5]:
df['job_posted_date'].dtype

dtype('<M8[ns]')

## Handling Missing Values 

In [6]:
df.isnull().sum()

job_title_short               0
job_title                     1
job_location               1045
job_via                       8
job_schedule_type         12667
job_work_from_home            0
search_location               0
job_posted_date               0
job_no_degree_mention         0
job_health_insurance          0
job_country                  49
salary_rate              752674
salary_year_avg          763738
salary_hour_avg          775079
company_name                 18
job_skills               117037
job_type_skills          117037
dtype: int64

### Checking for bias before dropping missing values
Before removing rows with missing values in non-critical columns, we verify these rows aren't concentrated in a specific segment (e.g. a particular country or remote status) that could skew the analysis.

In [7]:
cols_to_check = ['job_title', 'job_location', 'job_via', 'job_schedule_type', 'job_country', 'company_name']

for col in cols_to_check:
    print(f"--- Missing values in '{col}' ---")
    print(f"Total missing: {df[col].isnull().sum()} ({df[col].isnull().mean()*100:.2f}%)")
    
    # Distribution among rows with missing values in this column
    null_dist = df[df[col].isnull()]['job_country'].value_counts(normalize=True).head(5)
    # Overall distribution for comparison
    overall_dist = df['job_country'].value_counts(normalize=True).head(5)
    
    print("\nTop countries among missing rows:")
    print(null_dist)
    print("\nTop countries overall (for comparison):")
    print(overall_dist)
    print("\n")

--- Missing values in 'job_title' ---
Total missing: 1 (0.00%)

Top countries among missing rows:
Series([], Name: proportion, dtype: float64)

Top countries overall (for comparison):
job_country
United States     0.262561
India             0.065023
United Kingdom    0.051388
France            0.050811
Germany           0.035248
Name: proportion, dtype: float64


--- Missing values in 'job_location' ---
Total missing: 1045 (0.13%)

Top countries among missing rows:
job_country
United States     0.753831
South Korea       0.194444
Sudan             0.034483
United Kingdom    0.013410
India             0.001916
Name: proportion, dtype: float64

Top countries overall (for comparison):
job_country
United States     0.262561
India             0.065023
United Kingdom    0.051388
France            0.050811
Germany           0.035248
Name: proportion, dtype: float64


--- Missing values in 'job_via' ---
Total missing: 8 (0.00%)

Top countries among missing rows:
job_country
United States    0.

### Handling missing values without losing data

Rather than dropping all rows with missing values in non-critical fields, missing values are handled selectively:
- `job_schedule_type`: missing values are filled with `'Not Specified'`. This preserves the row while being transparent about the missing information (checked earlier and found to be slightly concentrated in certain countries, e.g. the Philippines).
- `job_location`, `job_via`, `company_name`, `job_title`: left as they are, since these fields are rarely used as core analysis variables and dropping them would cause unnecessary data loss.
- `job_skills` / `job_type_skills`: left untouched here. Missing values are filtered out only within analyses that specifically require skill data.

In [8]:
df['job_schedule_type'] = df['job_schedule_type'].fillna('Not Specified')

In [9]:
#Checking
df['job_schedule_type'].isnull().sum()

np.int64(0)

## Creating a Dedicated Salary Dataset

Since salary fields are only populated in ~4% of postings, they are isolated into a separate dataset (`df_salary`) rather than dropped from or imputed in the main dataset. This keeps the main dataset fully usable for non-salary analyses, while salary-specific analyses draw from a clean, complete subset.

In [10]:
df_salary = df[df['salary_year_avg'].notna() | df['salary_hour_avg'].notna()].copy()

print(f"Main dataset: {len(df):,} rows")
print(f"Salary dataset: {len(df_salary):,} rows ({len(df_salary)/len(df)*100:.2f}%)")

Main dataset: 785,741 rows
Salary dataset: 32,665 rows (4.16%)


## Standardizing Country Names & Verifying Remote Flag

In [11]:
df['job_country'].value_counts().head(20)

job_country
United States     206292
India              51088
United Kingdom     40375
France             39922
Germany            27694
Spain              25100
Singapore          23696
Sudan              21781
Netherlands        20631
Italy              17013
Canada             16029
Mexico             15139
Poland             14793
Portugal           14508
Australia          12955
South Africa       12414
Belgium            12078
Philippines        11786
Ireland            11162
Switzerland         9924
Name: count, dtype: int64

In [12]:
df[df['job_work_from_home'] == True]['job_location'].value_counts().head(10)

job_location
Anywhere    69552
Name: count, dtype: int64

**Findings**: No inconsistent country naming was found (e.g., no "USA" vs "United States" duplicates), so no standardization was needed. The remote work flag (`job_work_from_home`) was also confirmed reliable; every row flagged as remote had `job_location == 'Anywhere'`, with no exceptions. This validates `job_work_from_home` as the single source of truth for identifying remote roles throughout the analysis.

## Exporting the Cleaned Dataset

The cleaned dataset is exported to `data/clean/` as a reusable artifact for the analysis notebooks that follow, mirroring a lightweight data pipeline structure.

In [13]:
df.to_parquet('../data/clean/data_jobs_clean.parquet', index=False)
df_salary.to_parquet('../data/clean/data_jobs_salary.parquet', index=False)

print("Clean datasets exported successfully.")

Clean datasets exported successfully.


In [14]:
# Sanity check: reload clean data
df_check = pd.read_parquet('../data/clean/data_jobs_clean.parquet')
df_check.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 785741 entries, 0 to 785740
Data columns (total 17 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   job_title_short        785741 non-null  object        
 1   job_title              785740 non-null  object        
 2   job_location           784696 non-null  object        
 3   job_via                785733 non-null  object        
 4   job_schedule_type      785741 non-null  object        
 5   job_work_from_home     785741 non-null  bool          
 6   search_location        785741 non-null  object        
 7   job_posted_date        785741 non-null  datetime64[ns]
 8   job_no_degree_mention  785741 non-null  bool          
 9   job_health_insurance   785741 non-null  bool          
 10  job_country            785692 non-null  object        
 11  salary_rate            33067 non-null   object        
 12  salary_year_avg        22003 non-null   floa

## Summary

- Parsed `job_skills` and `job_type_skills` from string representations into usable Python lists/dictionaries.
- Converted `job_posted_date` to a proper `datetime` type for time-based analysis.
- Checked missing values column by column rather than dropping them blindly — found a small but real geographic bias in `job_location` (0.13% missing, concentrated in the US and South Korea) and `job_schedule_type` (1.6% missing, concentrated in the Philippines and Malaysia). Given the low volume involved, these were kept: `job_schedule_type` nulls were filled with `'Not Specified'` rather than dropped, preserving the rows while staying transparent about the gap.
- Isolated salary data into a dedicated `df_salary` subset (32,665 rows, ~4.2% of the dataset), since salary fields are only populated in a small minority of postings — this avoids either dropping 96% of the data or dragging incomplete columns through unrelated analyses.
- Validated `job_work_from_home` as a fully reliable remote work indicator: every row flagged as remote had `job_location == 'Anywhere'`, with no exceptions. Remote postings make up **8.85%** of all postings.
- Exported both the cleaned dataset and the salary subset to `.parquet`, preserving data types (dates, booleans, parsed lists/dicts) for reuse across analysis notebooks.